In [ ]:
from google.colab import drive

drive.mount("/content/drive")


# Coleta do Plenario da Camara

Notebook Colab para clonar o repositorio, instalar dependencias e executar o coletor `coleta.camara.plenario_discursos.collect` salvando dados no Google Drive.

A primeira celula monta o Drive antes de qualquer codigo do projeto. Os dados completos ficam em `MyDrive/falando_nela/data`; o clone do repositorio fica em `/content/falando_nela` e pode ser recriado a cada sessao.


In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)

for name in ["raw", "checkpoints", "logs", "manifests", "processed"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

print("FALANDO_NELA_DATA_ROOT=", os.environ["FALANDO_NELA_DATA_ROOT"])


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Repo em:", Path.cwd())
subprocess.run(["git", "status", "--short"], check=True)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


## Parametros

Use a validacao curta primeiro. Ela roda em modo producao para confirmar escrita no Drive, mas usa `--sample` para limitar a primeira particao anual e os primeiros deputados. A coleta completa fica protegida por uma flag. Para o backfill oficial de discursos do Plenario da Camara, o inicio operacional e `1946-01-01`; anos anteriores devem ser tratados como diagnostico separado, nao como corpus mensal default.


In [ ]:
from datetime import datetime, timezone

DATA_INICIO_VALIDACAO = "2026-05-01"
DATA_FIM_VALIDACAO = "2026-05-18"
SAMPLE_LIMIT_VALIDACAO = 3

DATA_INICIO_PROD = "1946-01-01"
DATA_FIM_PROD = "2026-05-18"

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID_VALIDACAO = f"colab-camara-plenario-validacao-{RUN_STAMP}"
RUN_ID_PROD = "prod-camara-plenario-baseline"

print("RUN_ID_VALIDACAO=", RUN_ID_VALIDACAO)
print("RUN_ID_PROD=", RUN_ID_PROD, "# fixo para permitir retomada com --resume")
print("Janela oficial de producao:", DATA_INICIO_PROD, "a", DATA_FIM_PROD)


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

MODULE = "coleta.camara.plenario_discursos.collect"


def run_collector(run_id, data_inicio, data_fim, sample=False, sample_limit=None, resume=True):
    cmd = [
        sys.executable,
        "-m",
        MODULE,
        "--mode",
        "prod",
        "--run-id",
        run_id,
        "--data-inicio",
        data_inicio,
        "--data-fim",
        data_fim,
    ]
    if resume:
        cmd.append("--resume")
    if sample:
        cmd.append("--sample")
    if sample_limit is not None:
        cmd.extend(["--sample-limit", str(sample_limit)])

    print("Rodando:", " ".join(cmd))
    output_lines = []
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line.rstrip("\n"))

    returncode = process.wait()
    manifest_path = manifest_from_output(output_lines, run_id)
    if returncode != 0:
        print(f"Coletor terminou com returncode={returncode}; veja logs/autosave.")
    return manifest_path


def manifest_from_output(output_lines, run_id):
    for line in reversed(output_lines):
        candidate = line.strip()
        if candidate.endswith(".json"):
            path = Path(candidate)
            if path.exists():
                return path

    manifest_path = DATA_ROOT / "manifests" / f"{run_id}.json"
    autosave_path = DATA_ROOT / "manifests" / f"{run_id}.autosave.json"
    if manifest_path.exists():
        return manifest_path
    if autosave_path.exists():
        return autosave_path
    return None


def show_manifest(manifest_path):
    if manifest_path is None:
        print("Manifest nao foi informado pelo coletor.")
        return None

    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        print("Manifest nao encontrado:", manifest_path)
        return None

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    keys = [
        "run_id",
        "source",
        "dataset",
        "mode",
        "sample",
        "sample_limit",
        "status",
        "errors",
        "deputados_processados",
        "paginas_discursos",
        "discursos",
        "discursos_com_transcricao",
        "preflight",
        "record_counts",
        "partition_counts",
        "processed_records_loaded",
        "log_path",
        "autosave_path",
    ]
    resumo = {key: manifest.get(key) for key in keys}
    print(json.dumps(resumo, ensure_ascii=False, indent=2))

    log_path = Path(manifest["log_path"])
    if log_path.exists():
        print("\nUltimas linhas do log:")
        print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-10:]))
    return manifest


def tail_log_for_run(run_id, n=30):
    log_path = DATA_ROOT / "logs" / f"{run_id}.jsonl"
    print("log_path=", log_path)
    if not log_path.exists():
        print("Log ainda nao encontrado.")
        return
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-n:]))


## Validacao curta

Esta celula roda em `--mode prod`, mas passa `--sample` para limitar a primeira particao e poucos deputados. Ela confirma que o Drive esta montado, que a escrita esta indo para `FALANDO_NELA_DATA_ROOT`, que as paginas de deputados ficam em `metadata/` e que as paginas de discursos ficam no JSONL mensal.


In [ ]:
manifest_validacao_path = run_collector(
    RUN_ID_VALIDACAO,
    DATA_INICIO_VALIDACAO,
    DATA_FIM_VALIDACAO,
    sample=True,
    sample_limit=SAMPLE_LIMIT_VALIDACAO,
    resume=True,
)
manifest_validacao = show_manifest(manifest_validacao_path)


In [ ]:
from datetime import date


def inspect_jsonl(path, max_rows=5):
    path = Path(path)
    print("\nArquivo:", path)
    if not path.exists():
        print("Nao encontrado")
        return

    lines = path.read_text(encoding="utf-8").splitlines()
    print("linhas=", len(lines), "bytes=", path.stat().st_size)
    for line in lines[:max_rows]:
        record = json.loads(line)
        payload = record.get("payload", {})
        discursos = payload.get("dados", []) if isinstance(payload, dict) else []
        first = discursos[0] if discursos and isinstance(discursos[0], dict) else {}
        print({
            "record_type": record.get("record_type"),
            "partition": record.get("partition"),
            "source_id": record.get("source_id"),
            "discursos_na_pagina": len(discursos),
            "dataHoraInicio": first.get("dataHoraInicio"),
            "tem_transcricao": bool(first.get("transcricao")),
            "tem_sumario": bool(first.get("sumario")),
        })


validation_start = date.fromisoformat(DATA_INICIO_VALIDACAO)
metadata_path = DATA_ROOT / "raw" / "camara" / "plenario_discursos" / "metadata" / f"{RUN_ID_VALIDACAO}.jsonl"
corpus_path = (
    DATA_ROOT
    / "raw"
    / "camara"
    / "plenario_discursos"
    / f"ano={validation_start.year:04d}"
    / f"mes={validation_start.month:02d}"
    / f"{RUN_ID_VALIDACAO}.jsonl"
)

inspect_jsonl(metadata_path, max_rows=10)
inspect_jsonl(corpus_path, max_rows=5)


## Coleta de madrugada

Esta celula roda o baseline inteiro em modo producao, com `run_id` fixo e `--resume`. Se o Colab cair ou a API falhar em alguma parte, rode a mesma celula novamente: o coletor le o checkpoint e os JSONLs ja gravados antes de continuar.


In [ ]:
EXECUTAR_MADRUGADA = False
RUN_ID_MADRUGADA = RUN_ID_PROD

if EXECUTAR_MADRUGADA:
    print("Rodando coleta de madrugada com run_id fixo:", RUN_ID_MADRUGADA)
    manifest_madrugada_path = run_collector(
        RUN_ID_MADRUGADA,
        DATA_INICIO_PROD,
        DATA_FIM_PROD,
        sample=False,
        resume=True,
    )
    manifest_madrugada = show_manifest(manifest_madrugada_path)
else:
    print("Altere EXECUTAR_MADRUGADA para True quando quiser iniciar a coleta completa.")


## Consultar coleta em andamento

Use esta celula em outra sessao do Colab para consultar manifesto, autosave e ultimas linhas de log do `run_id` fixo.


In [ ]:
autosave_madrugada_path = DATA_ROOT / "manifests" / f"{RUN_ID_MADRUGADA}.autosave.json"
manifest_madrugada_path = DATA_ROOT / "manifests" / f"{RUN_ID_MADRUGADA}.json"

if manifest_madrugada_path.exists():
    show_manifest(manifest_madrugada_path)
elif autosave_madrugada_path.exists():
    show_manifest(autosave_madrugada_path)
else:
    print("Ainda nao ha manifest/autosave para", RUN_ID_MADRUGADA)

tail_log_for_run(RUN_ID_MADRUGADA, n=30)
